In [ ]:
rows = []
bm = qt['BM_ret'].fillna(0)

for strat, label in [('MOM','モメンタム'),('VAL','バリュー'),('QUAL','クオリティ'),('COMP','複合'),('BM','日経225')]:
    col  = 'BM_ret' if strat == 'BM' else f'{strat}_ret'
    rets = qt[col].fillna(0)
    cum  = (1 + rets).prod() - 1
    cagr = (1 + cum) ** (4/len(rets)) - 1
    alpha_yr = (rets.mean() - bm.mean()) * 4
    sharpe   = rets.mean() / rets.std() * np.sqrt(4) if rets.std() > 0 else np.nan
    cum_c    = (1 + rets).cumprod()
    max_dd   = ((cum_c - cum_c.cummax()) / cum_c.cummax()).min()
    win_rate = ((rets.values - bm.values) > 0).mean() if strat != 'BM' else np.nan

    rows.append({
        '戦略':           label,
        '累計リターン':    f"{cum*100:.1f}%",
        '年率リターン':    f"{cagr*100:.1f}%",
        '年率アルファ':    f"{alpha_yr*100:+.1f}%" if strat != 'BM' else '—',
        'シャープレシオ':  f"{sharpe:.2f}",
        '最大DD':         f"{max_dd*100:.1f}%",
        '勝率(対BM)':     f"{win_rate*100:.1f}%" if not np.isnan(win_rate) else '—',
    })

summary = pd.DataFrame(rows)
print(summary.to_string(index=False))
print("\n⚠️  VAL・QUALは現時点財務データ使用（ルックアヘッドバイアスあり）")
print("⚠️  MOM・COMPの価格部分はバイアスなし")

## サマリー（戦略別評価指標）

In [ ]:
COLORS = {'MOM':'#5c73f2','VAL':'#f59e0b','QUAL':'#06b6d4','COMP':'#22c55e','BM':'#94a3b8'}
LABELS = {'MOM':'モメンタム','VAL':'バリュー','QUAL':'クオリティ','COMP':'複合','BM':'日経225'}

fig, axes = plt.subplots(2, 1, figsize=(14, 10))
fig.patch.set_facecolor('#0f1117')

# ── 上段: 累積リターン ──
ax = axes[0]
ax.set_facecolor('#1a1d27')
x = range(len(qt))
for strat in ['MOM','VAL','QUAL','COMP','BM']:
    col = 'BM_ret' if strat == 'BM' else f'{strat}_ret'
    cum = (1 + qt[col].fillna(0)).cumprod() - 1
    lw = 2.5 if strat == 'COMP' else 1.5
    ls = '--' if strat == 'BM' else '-'
    ax.plot(x, cum*100, label=LABELS[strat], color=COLORS[strat], linewidth=lw, linestyle=ls)

ax.axhline(0, color='#2e3248', linewidth=1)
ax.set_xticks(list(x)[::4])
ax.set_xticklabels(qt['quarter'].tolist()[::4], rotation=45, ha='right', fontsize=8, color='#94a3b8')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda v,_: f'{v:.0f}%'))
ax.tick_params(colors='#94a3b8')
ax.set_title('累積リターン推移（2015〜2024）', color='#e2e8f0', fontsize=13, pad=10)
ax.legend(facecolor='#232635', labelcolor='#e2e8f0', edgecolor='#2e3248')
ax.grid(True, color='#2e3248', linewidth=0.5, alpha=0.6)
for sp in ax.spines.values(): sp.set_edgecolor('#2e3248')

# ── 下段: 複合戦略の対BM超過リターン ──
ax2 = axes[1]
ax2.set_facecolor('#1a1d27')
excess = (qt['COMP_ret'].fillna(0) - qt['BM_ret'].fillna(0)) * 100
colors = ['#22c55e' if v >= 0 else '#ef4444' for v in excess]
ax2.bar(x, excess, color=colors, width=0.7)
ax2.axhline(0, color='#2e3248', linewidth=1)
ax2.set_xticks(list(x)[::4])
ax2.set_xticklabels(qt['quarter'].tolist()[::4], rotation=45, ha='right', fontsize=8, color='#94a3b8')
ax2.yaxis.set_major_formatter(mticker.FuncFormatter(lambda v,_: f'{v:.1f}%'))
ax2.tick_params(colors='#94a3b8')
ax2.set_title('複合戦略 対日経225 超過リターン（四半期別）', color='#e2e8f0', fontsize=13, pad=10)
ax2.grid(True, color='#2e3248', linewidth=0.5, alpha=0.6, axis='y')
for sp in ax2.spines.values(): sp.set_edgecolor('#2e3248')

plt.tight_layout(pad=3)
plt.show()

## 結果グラフ

In [ ]:
N_STOCKS = 10  # ← 保有銘柄数（変更可）

quarters = pd.date_range(start=TEST_START, end=TEST_END, freq='QS')
records  = []

for i, buy in enumerate(quarters):
    sell  = buy + pd.DateOffset(months=3)
    label = f"{buy.year}-Q{(buy.month-1)//3+1}"

    mom_s  = momentum_score(prices, buy)
    val_s  = value_score(fundamentals)
    qual_s = quality_score(fundamentals)
    comp_s = composite_score(prices, fundamentals, buy)

    row = {'quarter': label, 'buy': buy, 'sell': sell}
    for name, scores in [('MOM',mom_s),('VAL',val_s),('QUAL',qual_s),('COMP',comp_s)]:
        picks = select_top(scores, N_STOCKS)
        row[f'{name}_ret']     = quarter_return(prices, picks, buy, sell)
        row[f'{name}_tickers'] = picks

    row['BM_ret'] = quarter_return(prices, [BENCHMARK], buy, sell)
    records.append(row)
    print(f"  {label} | MOM:{row['MOM_ret']*100:+5.1f}%  COMP:{row['COMP_ret']*100:+5.1f}%  BM:{row['BM_ret']*100:+5.1f}%")

qt = pd.DataFrame(records)
print(f"\n✅ バックテスト完了: {len(qt)}クォーター")

## バックテスト実行

In [ ]:
def zscore(s):
    return (s - s.mean()) / s.std() if s.std() > 0 else s * 0

def momentum_score(prices, date):
    """12-1ヶ月モメンタム。価格データのみ・バイアスなし。"""
    end   = date - pd.DateOffset(months=1)
    start = date - pd.DateOffset(months=13)
    w = prices.loc[start:end].ffill()
    if len(w) < 20: return pd.Series(dtype=float)
    return ((w.iloc[-1] / w.iloc[0]) - 1).dropna()

def value_score(fundamentals):
    """低PBR戦略。⚠️ 現時点財務データ使用。"""
    pb = {t: v['priceToBook'] for t, v in fundamentals.items()
          if v.get('priceToBook') and v['priceToBook'] > 0}
    return -pd.Series(pb)  # 低いほど良い

def quality_score(fundamentals):
    """高ROE・低負債戦略。⚠️ 現時点財務データ使用。"""
    scores = {}
    for t, v in fundamentals.items():
        roe = v.get('returnOnEquity')
        de  = v.get('debtToEquity')
        if roe is None or np.isnan(roe): continue
        scores[t] = roe - (0.3 * np.log1p(de) if de and de > 0 else 0)
    return pd.Series(scores)

def composite_score(prices, fundamentals, date):
    """MOM×0.5 + VAL×0.25 + QUAL×0.25"""
    mom  = zscore(momentum_score(prices, date))
    val  = zscore(value_score(fundamentals))
    qual = zscore(quality_score(fundamentals))
    idx  = set(mom.index)
    df   = pd.DataFrame({'mom': mom, 'val': val, 'qual': qual}).reindex(list(idx)).fillna(0)
    return df['mom']*0.5 + df['val']*0.25 + df['qual']*0.25

def select_top(scores, n=10):
    return scores.nlargest(n).index.tolist() if not scores.empty else []

def quarter_return(prices, tickers, buy, sell):
    rets = []
    for t in tickers:
        if t not in prices.columns: continue
        w = prices.loc[buy:sell, t].dropna()
        if len(w) < 2: continue
        rets.append(w.iloc[-1] / w.iloc[0] - 1)
    return float(np.mean(rets)) if rets else np.nan

print("✅ 戦略定義完了")

## 戦略の定義（学習なし・固定ルール）

In [ ]:
# ── データ取得 ──
all_tickers = TICKERS + [BENCHMARK]
prices = yf.download(all_tickers, start="2014-01-01", end="2025-01-01",
                     auto_adjust=True, progress=False)['Close']
prices.columns = [str(c) for c in prices.columns]
prices.index   = pd.to_datetime(prices.index)

# ── 財務データ取得 ──
print("財務データを取得中...")
fundamentals = {}
for t in TICKERS:
    try:
        info = yf.Ticker(t).info
        fundamentals[t] = {
            'priceToBook':    info.get('priceToBook'),
            'returnOnEquity': info.get('returnOnEquity'),
            'debtToEquity':   info.get('debtToEquity'),
        }
    except:
        fundamentals[t] = {}

print(f"✅ 株価データ: {prices.shape[1]}銘柄 × {len(prices)}営業日")
print(f"✅ 財務データ: {len(fundamentals)}銘柄")

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import yfinance as yf
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

# ── 銘柄リスト ──
TICKERS = [
    "6758.T","6861.T","8035.T","6954.T","6971.T","6702.T","6501.T","6503.T","6902.T","6594.T",
    "7203.T","7267.T","7270.T","7011.T",
    "9984.T","9432.T","9433.T","6098.T",
    "8306.T","8316.T","8058.T","8031.T","8001.T",
    "4063.T","4901.T","3407.T","5401.T","5802.T","5108.T",
    "4519.T","4568.T","4502.T","4523.T",
    "9983.T","3382.T","8267.T","2914.T","2802.T","4911.T",
    "8802.T","9020.T","9022.T","4661.T",
    "7741.T","7733.T","4543.T",
]
BENCHMARK  = "^N225"
TEST_START = "2015-01-01"
TEST_END   = "2024-10-01"

print(f"銘柄数: {len(TICKERS)}")
print("データ取得を開始します（数分かかります）...")

In [ ]:
!pip install yfinance -q

# 日本株 戦略バックテストシステム
**対象:** 日経225主要構成銘柄  
**期間:** 2015〜2024年（40クォーター）  
**戦略:** モメンタム / バリュー / クオリティ / 複合  
**ルール:** 学習なし・フィードバックなし・固定ルールのみ